        МОНИТОРИНГ КАЧЕСТВА ДАННЫХ И МОДЕЛЕЙ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

    1. Загрузка логов проверок качества данных и мониторинга моделей

In [ ]:
df_checks = pd.read_csv('data_quality_checks_log.csv')
df_model = pd.read_csv('model_monitoring.csv')

df_checks['actual_value'] = df_checks['actual_value'].fillna(df_checks['actual_value'].mean())

df_checks['execution_datetime '] = pd.to_datetime(df_checks['execution_datetime'])
df_model['monitoring_date'] = pd.to_datetime(df_model['monitoring_date'])
df_model['monitoring_time'] = pd.to_datetime(df_model['monitoring_time'])

    2. Анализ логов проверок качества (DQ Checks Log)

In [ ]:
print("=== ИНФОРМАЦИЯ О ЛОГАХ ПРОВЕРОК DQ ===")
print(df_checks.info())
print("\n=== ПЕРВЫЕ СТРОКИ (DQ CHECKS) ===")
print(df_checks.head())

    3. Анализ показателей мониторинга моделей (Model Monitoring)

In [ ]:
print("\n=== ИНФОРМАЦИЯ О МОНИТОРИНГЕ МОДЕЛЕЙ ===")
print(df_model.info())
print("\n=== ОПИСАТЕЛЬНАЯ СТАТИСТИКА МОДЕЛЕЙ ===")
print(df_model.describe())

    4. Показатели мониторинга по моделям

In [ ]:
list_of_metrics = ['accuracy', 'precision', 'recall', 'auc_roc', 'gini', 'f1_score']
df_model['gini'] = (2 * df_model['auc_roc'] - 1)
matrix_of_correlation_metrics = df_model[list_of_metrics].corr()

model_metrics_grouped_by_model_name = df_model.groupby(by= ['model_name', 'model_type'])[list_of_metrics]
model_metrics_grouped_by_model_name.columns = ['model_name', 'model_type'] + list_of_metrics

print("Количество наблюдений по моделям.\n", model_metrics_grouped_by_model_name.count(), "\n\n")  

print("Среднее по метрикам.\n", model_metrics_grouped_by_model_name.mean().round(3), "\n\n") 

print("Медиана метрик по моделям: \n", model_metrics_grouped_by_model_name.median().round(3), "\n\n")

    5. Расчет корреляции "data_drift_score VS accuracy"

In [ ]:
print("Корреляция data_drift_score с accuracy \n", df_model[['data_drift_score', 'accuracy']].corr(), "\n\n")
list_metriks_and_dates_names = ['data_drift_score'] + list_of_metrics
# 1 - 
model_metrics_grouped_by_date_nameOfmodel = (df_model
                                             .groupby(by= ['monitoring_date', 'model_name'])[list_metriks_and_dates_names]
                                             .mean().sort_values('monitoring_date')
                                             .reset_index())
model_metrics_grouped_by_date_nameOfmodel.columns = ['monitoring_date', 'model_name'] + list_metriks_and_dates_names


# 2 -
ml_metr_group_date_nm = model_metrics_grouped_by_date_nameOfmodel
model_metrics_grouped_by_date = (ml_metr_group_date_nm.groupby(by= 'monitoring_date')
                                 [list_metriks_and_dates_names]
                                 .mean()
                                 .sort_values('monitoring_date'))
table_Pivot= model_metrics_grouped_by_date

date_name= 'monitoring_date'


for i in list_metriks_and_dates_names[0: 2]:
    fig, (ax1) = plt.subplots(1, 1, figsize= (16, 4))
    vl_nm_2 = i
    ax1.plot(table_Pivot[date_name], table_Pivot[vl_nm_2], marker= 'o',
                linewidth= 2, markersize= 4, color= 'red', linestyle= '-')
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y'))
    ax1.xaxis.set_major_locator(mdates.DayLocator(interval= 7)) # Интервал в днях - Чисто по Ох отметить, на апроксимацию не влияет.
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation= 45, ha= 'right') # Поворот подписей дат, условно.
    ax1.grid(True, alpha=0.3, linestyle='--')
    ax1.set_title(f'Динамика {i} по неделям', fontsize=16, pad=20)
    ax1.set_xlabel('Дата наблюдения')
    ax1.set_ylabel(f'Значение среднего {i}')

    plt.tight_layout()
    plt.show()
    

# plt.savefig(f'Plot_.png', format='png', dpi=300, bbox_inches='tight')  